In [60]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

 
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.tree import export_text



In [61]:

data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
!wget $data -O car_fuel_efficiency.csv

df = pd.read_csv('car_fuel_efficiency.csv')
df.head()

--2025-11-03 19:36:47--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.04s   

2025-11-03 19:36:47 (21.3 MB/s) - ‘car_fuel_efficiency.csv’ saved [874188/874188]



,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [62]:
# fill missing values with 0
df.fillna(0, inplace=True)


In [63]:
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,0.0,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,0.0,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [64]:
# setup train, val, test splits
# df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
# df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
# df_train = df_train.reset_index(drop=True)
# df_val = df_val.reset_index(drop=True)
# df_test = df_test.reset_index(drop=True)

 
# y_train = df_train.fuel_efficiency_mpg.values
# y_val = df_val.fuel_efficiency_mpg.values
# y_test = df_test.fuel_efficiency_mpg.values
 
# del df_train['fuel_efficiency_mpg']
# del df_val['fuel_efficiency_mpg']
# del df_test['fuel_efficiency_mpg'] 

In [65]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# Define your target column
target = 'fuel_efficiency_mpg'

# Separate features and target for each split
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_val = df_val.drop(columns=[target])
y_val = df_val[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]


In [66]:
# Identify numeric and categorical columns
numeric_features = ["engine_displacement", "num_cylinders", "horsepower",
                    "vehicle_weight", "acceleration", "model_year", "num_doors"]

categorical_features = ["origin", "fuel_type", "drivetrain"]

In [67]:
# remove categorial columns 
df_val = df_val.drop(columns=categorical_features)
df_train = df_train.drop(columns=categorical_features)
df_test = df_test.drop(columns=categorical_features)

In [68]:
df_train.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,num_doors,fuel_efficiency_mpg
0,120,5.0,169.0,2966.679505,13.9,2005,-1.0,15.301475
1,200,3.0,143.0,2950.822121,17.1,2013,-1.0,15.331215
2,180,6.0,180.0,3078.221669,17.4,2007,0.0,15.336679
3,280,5.0,174.0,2797.991793,0.0,2016,0.0,15.865850
4,250,4.0,133.0,2362.426930,16.3,2010,-1.0,18.102203


In [69]:
# one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # Keep numeric columns as they are
)


In [70]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?

In [71]:

train_dicts = df_train.fillna(0).to_dict(orient='records')
train_dicts[:5]
dv = DictVectorizer(sparse=True)
dt = DecisionTreeRegressor(random_state=1, max_depth=1)

# keep dv fitted 
X_train = dv.fit_transform(train_dicts)
dt.fit(X_train, y_train)

model = DecisionTreeRegressor(random_state=1, max_depth=1)

model.fit(X_train, y_train)




,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [72]:
# print(export_text(dt))
names = dv.get_feature_names_out().tolist()
print(export_text(dt, feature_names=names))

|--- fuel_efficiency_mpg <= 14.96
|   |--- value: [12.94]
|--- fuel_efficiency_mpg >  14.96
|   |--- value: [16.98]



In [73]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?
# vehicle_weight

In [74]:
# Question 2
# Train a random forest regressor with these parameters:

# n_estimators=10
# random_state=1
# n_jobs=-1 (optional - to make training faster)
# What's the RMSE of this model on the validation data?
# 9.36 4.5


In [75]:
model = RandomForestRegressor(n_estimators=10, random_state=1,n_jobs=-1)
feature_names = dv.get_feature_names_out()

model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred = model.predict(df_val)


# Compute RMSE
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(f"RMSE on validation data: {rmse:.2f}")


RMSE on validation data: 9.36


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [76]:
# Question 3
# Now let's experiment with the n_estimators parameter

# Try different values of this parameter from 10 to 200 with step 10.
# Set random_state to 1.
# Evaluate the model on the validation dataset.

In [77]:
estimators = list(range(10, 201, 10))
 
for e in estimators:
    model = RandomForestRegressor(n_estimators=e, random_state=1,n_jobs=-1)

    model.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred = model.predict(df_val)

    # Compute RMSE
    mse = mean_squared_error(y_val, y_pred)

    rmse = np.sqrt(mse)

    print(f"RMSE on validation data: {rmse:.3f}", e)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.360 10


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.364 20


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


...

/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.387 180


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.379 190
RMSE on validation data: 9.402 200


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [78]:
#  Question 4
# Let's select the best max_depth:

# Try different values of max_depth: [10, 15, 20, 25]
# For each of these values,
# try different values of n_estimators from 10 till 200 (with step 10)
# calculate the mean RMSE
# Fix the random seed: random_state=1
# What's the best max_depth, using the mean RMSE?
# Answer 10 4.22

In [79]:
estimators = list(range(10, 201, 10))
depths = [10, 15, 20, 25]
 
for d in depths:
  for e in estimators:
      model = RandomForestRegressor(n_estimators=e, random_state=1,n_jobs=-1)

      model.fit(X_train, y_train)

      # Make predictions on the validation set
      y_pred = model.predict(df_val)

      # Compute RMSE
      mse = mean_squared_error(y_val, y_pred)

      rmse = np.sqrt(mse)

      print(f"RMSE on validation data: {rmse:.3f}", d, e)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.360 10 10


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.364 10 20


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


...

/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.387 25 180


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


RMSE on validation data: 9.379 25 190
RMSE on validation data: 9.402 25 200


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [ ]:
# Question 5
# Train the model with these parameters:
# n_estimators=10,
# max_depth=20,
# random_state=1,
# n_jobs=-1 (optional)
# Get the feature importance information from this model
# What's the most important feature (among these 4)?

# vehicle_weight
# horsepower
# acceleration
# engine_displacement
# answer acceleration

In [82]:
model = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1,n_jobs=-1)

model.fit(X_train, y_train)
# Get feature importances
importances = model.feature_importances_
print(importances)
feature_names = ['vehicle_weight',
                'horsepower',
                'acceleration',
                'engine_displacement']
print(feature_names)

# Print feature importances matched to names
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

[2.55267172e-05 1.64970155e-05 9.99760281e-01 1.59264435e-05
 3.53902604e-06 1.14863410e-06 2.56376210e-05 1.51443294e-04]
['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
vehicle_weight: 0.0000
horsepower: 0.0000
acceleration: 0.9998
engine_displacement: 0.0000
